In [ ]:
# Basic QXYCell workflow
#
# Replace the example path below with your own QuPath export/input folder.
#
# Expected input folder contents:
# - measurement TSV files exported from QuPath
# - optional GeoJSON annotation files
# - optional thresholds table or object classifier JSON files
# - optional core/sample metadata tables
#
# Output workflow:
# - qxy.check() creates <project_dir.name>_check_YYMMDD_HHMM beside project_dir.
# - qxy.run() creates <project_dir.name>_run_YYMMDD_HHMM beside project_dir.
# - Threshold tables are saved in a sibling thresholds/ folder beside project_dir.
# - Downstream steps reuse adata.uns["qxycell"]["output_dir"] by default.

from pathlib import Path

import pandas as pd
import qxycell as qxy


In [ ]:
# 1. Set project path.
# project_dir is the QuPath export/input folder.

project_dir = Path("path/to/qupath_export").expanduser().resolve()


In [ ]:
# 2. Inspect available input files.

input_files = sorted(
    p.relative_to(project_dir)
    for p in project_dir.rglob('*')
    if p.is_file()
)

input_files[:50]

In [ ]:
# 3. Optional: generate a threshold table from object classifiers.
# This creates a fresh timestamped table in project_dir.parent / "thresholds".
# It does not run check(), modify an existing threshold table, or modify AnnData.
# Skip this cell if you already have a thresholds.tsv / thresholds_*.tsv table.

threshold_template = qxy.generate_threshold_table(project_dir)
threshold_template


In [ ]:
# 4. Select the active threshold table.
#
# Leave threshold_file as None to use the newest recognized table found in either
# project_dir or project_dir.parent / "thresholds". To force a specific table,
# set threshold_file to that path.

threshold_file = None


In [ ]:
# 5. Run QXYCell check.
# qxy.check() validates measurements, threshold definitions, and GeoJSON files.
# It writes only check outputs to <project_dir.name>_check_YYMMDD_HHMM.
# It does not generate threshold tables.

report = qxy.check(
    project_dir,
    threshold_file=threshold_file,
    count_rows=True,
)

report.ok


In [ ]:
# Inspect check messages.

pd.DataFrame([message.to_dict() for message in report.messages])

In [ ]:
# 6. Import QuPath data to AnnData.
# qxy.run() creates <project_dir.name>_run_YYMMDD_HHMM beside project_dir.
# It does not call qxy.check() or write a check report.
# If no threshold table exists, run() generates one from object classifiers in
# project_dir.parent / "thresholds" before import.
# It does not create marker positivity columns or cell types by default.

adata = qxy.run(
    project_dir,
    threshold_file=threshold_file,
    pixel_size_um=0.28,
)

output_dir = Path(adata.uns["qxycell"]["output_dir"])
print(output_dir)
adata


In [ ]:
# Preview imported observation metadata and marker measurement mapping.

obs_preview_columns = [
    column
    for column in ['Image', 'Object ID', 'Centroid X um', 'Centroid Y um', 'CoreID']
    if column in adata.obs.columns
]

var_preview_columns = [
    column
    for column in ['marker_name', 'source_measurement_column', 'threshold']
    if column in adata.var.columns
]

display(adata.obs[obs_preview_columns].head())
display(adata.var[var_preview_columns].head())

In [ ]:
# 7. Apply marker thresholds.
# qxy.threshold() applies the active threshold table to an existing AnnData
# object and writes <marker>_pos columns to adata.obs.

threshold_summary = qxy.threshold(
    adata,
    project_dir,
    threshold_file=threshold_file,
)

threshold_summary


In [ ]:
# Preview positivity columns.

pos_columns = sorted(column for column in adata.obs.columns if column.endswith('_pos'))
adata.obs[pos_columns].head()

In [ ]:
# 8. Optional: remove ignored cells before downstream summaries.
# This uses GeoJSON annotation columns imported by qxy.run().

if 'Ignore' in adata.obs.columns:
    adata = qxy.remove_ignore(adata)

adata

In [ ]:
# 9. Optional: add sample metadata.
# Pass a project metadata CSV/TSV path or DataFrame to qxy.add_metadata().
# sample_col is the adata.obs column to match, and metadata_sample_col is the
# matching column in the metadata table.

# Example:
# metadata_table = project_dir / "sample_metadata.tsv"
# metadata_summary = qxy.add_metadata(
#     adata,
#     metadata_table,
#     sample_col="Image",
#     metadata_sample_col="Image",
# )
# metadata_summary


In [ ]:
# 10. Optional: identify cores with missing metadata.
# Replace required_metadata_columns with the metadata fields expected for your
# project after metadata has been added.

required_metadata_columns = []

if 'CoreID' in adata.obs.columns and required_metadata_columns:
    missing_metadata_by_core = (
        adata.obs.groupby('CoreID', observed=True)[required_metadata_columns]
        .apply(lambda frame: frame.isna().any().any())
        .rename('has_missing_metadata')
        .reset_index()
    )
    missing_metadata_by_core = missing_metadata_by_core[
        missing_metadata_by_core['has_missing_metadata']
    ]
else:
    missing_metadata_by_core = pd.DataFrame(columns=['CoreID', 'has_missing_metadata'])

missing_metadata_by_core

In [ ]:
# 11. Create a cell type logic template.
# Review the printed prompt, then save your edited YAML and pass it to qxy.celltype().

celltype_prompt = qxy.celltype_prompt(adata)
print(celltype_prompt)

In [ ]:
# 12. Apply cell type logic.
# Uncomment after saving a reviewed YAML file.

# celltype_logic = Path('celltype_logic.yaml')
# celltype_summary = qxy.celltype(adata, celltype_logic, column='celltype')
# celltype_summary

In [ ]:
# 13. Run QC and plot quick spatial previews.

qc_summary = qxy.qc(adata)
qc_summary


In [ ]:
# 14. Save the final AnnData object.

h5ad_path = qxy.save(adata)
h5ad_path
